In [0]:
import json
from databricks.sdk import WorkspaceClient

w = WorkspaceClient()
host = spark.conf.get("spark.databricks.workspaceUrl")

JOB_NAME = "data_sentinel_agent1"
ROOT = "/Workspace/Users/s819534@8451.com/Shashi-data-sentinel-barrows-dev/data_sentinel_barrows"
PIPE_PATH = f"{ROOT}/agent1/pipeline"

BASELINE_PATH = f"{PIPE_PATH}/01_baseline_computation"
VALIDATION_PATH = f"{PIPE_PATH}/02_validation_agent"
AI_SUMM_PATH = f"{PIPE_PATH}/03_ai_summarization"
ALERTS_PATH = f"{PIPE_PATH}/04_teams_alert_dispatcher"

NOTIFICATION_EMAILS = ["s819534@8451.com"]

print("=" * 60)
print("AGENT 1 - WORKFLOW CREATE/RESET")
print("=" * 60)
print(f"Workspace host : {host}")
print(f"Job name       : {JOB_NAME}")
print(f"Root path      : {ROOT}")
print("Notebook paths:")
print(f"  Baseline     : {BASELINE_PATH}")
print(f"  Validation   : {VALIDATION_PATH}")
print(f"  AI Summary   : {AI_SUMM_PATH}")
print(f"  Alerts       : {ALERTS_PATH}")
print("=" * 60)

## Step 1: Verify Notebooks Exist Before Creating Workflow

In [0]:
def verify_notebook_exists(path: str) -> bool:
    response = w.api_client.do(
        "GET", "/api/2.0/workspace/get-status", query={"path": path}
    )
    return bool(response)

paths = [BASELINE_PATH, VALIDATION_PATH, AI_SUMM_PATH, ALERTS_PATH]

print("Verifying notebook paths...")
missing = []
for path in paths:
    try:
        ok = verify_notebook_exists(path)
    except Exception:
        ok = False
    status = "EXISTS  OK" if ok else "MISSING"
    print(f"  {status:12} {path}")
    if not ok:
        missing.append(path)

if missing:
    raise RuntimeError(
        "Missing notebook paths:\n" + "\n".join(missing)
    )

print("\nAll notebooks verified.")

## Step 2: Define the Workflow

In [0]:
job_settings = {
    "name": JOB_NAME,
    "schedule": {
        "quartz_cron_expression": "0 30 14 * * ?",
        "timezone_id": "UTC",
        "pause_status": "PAUSED",
    },
    "email_notifications": {
        "on_failure": NOTIFICATION_EMAILS,
        "no_alert_for_skipped_runs": True,
    },
    "max_concurrent_runs": 1,
    "queue": {"enabled": True},
    "parameters": [
        {"name": "run_mode", "default": "SCHEDULED"},
        {"name": "table_filter", "default": ""},
        {"name": "as_of_date", "default": ""},
        {"name": "baseline_start", "default": ""},
        {"name": "baseline_end", "default": ""},
        {"name": "business_date", "default": ""},
        {"name": "baseline_version_id", "default": ""},
        {"name": "run_validation", "default": "true"},
        {"name": "window_days", "default": "30"},
        {"name": "window_policy", "default": "MONTH"},
        {"name": "refresh_days", "default": "1,15"},
        {"name": "catalog", "default": "learning_dev"},
        {"name": "schema", "default": "s819534"},
    ],
    "tasks": [
        {
            "task_key": "cond_baseline",
            "condition_task": {
                "left": "{{job.parameters.run_mode}}",
                "op": "EQUAL_TO",
                "right": "VALIDATE_ONLY",
            },
            "timeout_seconds": 3600,
        },
        {
            "task_key": "baseline",
            "depends_on": [{"task_key": "cond_baseline", "outcome": "false"}],
            "notebook_task": {
                "notebook_path": BASELINE_PATH,
                "source": "WORKSPACE",
                "base_parameters": {
                    "run_mode": "{{job.parameters.run_mode}}",
                    "table_filter": "{{job.parameters.table_filter}}",
                    "baseline_start": "{{job.parameters.baseline_start}}",
                    "baseline_end": "{{job.parameters.baseline_end}}",
                    "window_days": "{{job.parameters.window_days}}",
                    "window_policy": "{{job.parameters.window_policy}}",
                    "refresh_days": "{{job.parameters.refresh_days}}",
                    "as_of_date": "{{job.parameters.as_of_date}}",
                    "catalog": "{{job.parameters.catalog}}",
                    "schema": "{{job.parameters.schema}}",
                    "job_run_id": "{{job.run_id}}",
                },
            },
            "timeout_seconds": 3600,
            "max_retries": 1,
            "email_notifications": {"on_failure": NOTIFICATION_EMAILS},
        },
        {
            "task_key": "cond_validation",
            "depends_on": [{"task_key": "cond_baseline"}, {"task_key": "baseline"}],
            "run_if": "ALL_DONE",
            "condition_task": {
                "left": "{{job.parameters.run_validation}}",
                "op": "EQUAL_TO",
                "right": "true",
            },
            "timeout_seconds": 3600,
        },
        {
            "task_key": "validation",
            "depends_on": [{"task_key": "cond_validation", "outcome": "true"}],
            "notebook_task": {
                "notebook_path": VALIDATION_PATH,
                "source": "WORKSPACE",
                "base_parameters": {
                    "business_date": "{{job.parameters.business_date}}",
                    "run_validation": "{{job.parameters.run_validation}}",
                    "table_filter": "{{job.parameters.table_filter}}",
                    "baseline_version_id": "{{job.parameters.baseline_version_id}}",
                    "catalog": "{{job.parameters.catalog}}",
                    "schema": "{{job.parameters.schema}}",
                    "job_run_id": "{{job.run_id}}",
                },
            },
            "timeout_seconds": 3600,
            "max_retries": 1,
            "email_notifications": {"on_failure": NOTIFICATION_EMAILS},
        },
        {
            "task_key": "cond_ai_summary",
            "depends_on": [{"task_key": "validation"}],
            "run_if": "ALL_DONE",
            "condition_task": {
                "left": "{{tasks.validation.values.validation_run_ids}}",
                "op": "EQUAL_TO",
                "right": "[]",
            },
            "timeout_seconds": 3600,
        },
        {
            "task_key": "ai_summary",
            "depends_on": [{"task_key": "cond_ai_summary", "outcome": "false"}],
            "notebook_task": {
                "notebook_path": AI_SUMM_PATH,
                "source": "WORKSPACE",
                "base_parameters": {
                    "catalog": "{{job.parameters.catalog}}",
                    "schema": "{{job.parameters.schema}}",
                    "validation_run_ids": "{{tasks.validation.values.validation_run_ids}}",
                },
            },
            "timeout_seconds": 3600,
        },
        {
            "task_key": "alerts",
            "depends_on": [{"task_key": "ai_summary"}],
            "notebook_task": {
                "notebook_path": ALERTS_PATH,
                "source": "WORKSPACE",
                "base_parameters": {
                    "catalog": "{{job.parameters.catalog}}",
                    "schema": "{{job.parameters.schema}}",
                },
            },
            "timeout_seconds": 3600,
        },
    ],
}

print("Workflow settings prepared.")
print(f"Tasks defined: {len(job_settings['tasks'])}")
for task in job_settings["tasks"]:
    print(f"  - {task['task_key']}")

## Step 3: Create the Workflow

In [0]:
def find_job_id_by_name(name: str):
    response = w.api_client.do("GET", "/api/2.1/jobs/list", query={"name": name})
    jobs = response.get("jobs", []) if isinstance(response, dict) else []
    for job in jobs:
        settings = job.get("settings", {})
        if settings.get("name") == name:
            return job.get("job_id")
    return None

job_id = find_job_id_by_name(JOB_NAME)
if job_id:
    print(f"Found existing job '{JOB_NAME}' (job_id={job_id}). Resetting...")
    w.api_client.do(
        "POST", "/api/2.1/jobs/reset",
        body={"job_id": int(job_id), "new_settings": job_settings},
    )
else:
    print(f"Job '{JOB_NAME}' not found. Creating...")
    created = w.api_client.do("POST", "/api/2.1/jobs/create", body=job_settings)
    job_id = created.get("job_id")

job_url = f"https://{host}/#job/{job_id}"
print("\nCreate-or-reset complete.")
print(f"job_id: {job_id}")
print(f"job_url: {job_url}")
print("Job was NOT run.")

## Step 4: Trigger a Manual Test Run

In [0]:
print("Manual run trigger removed by design.")
print("This notebook only creates/resets the workflow and does NOT run it.")
print("Use Workflows UI or jobs/run-now explicitly when needed.")

## Step 5: Monitor the Run

In [0]:
print("Run monitoring removed in this creator notebook.")
print("No run is started here, so there is nothing to monitor.")

## Step 6: Enable the Schedule (Run After Successful Test)

In [0]:
print("Schedule is set to PAUSED by default in job settings.")
print("No post-create mutation is performed here.")

## Step 7: Verify End-to-End Results

In [0]:
print("End-to-end result query removed from workflow creator.")
print("Use dedicated validation/report notebooks after workflow runs.")